In [1]:
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any, Optional

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import NearestNeighbors

from scipy.stats import norm, qmc
from scipy.optimize import minimize

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# WEEK 10 — FUNCTION 8 (TRANSPARENCY/INTERPRETABILITY v4)
# Upgrades vs Week 9:
#  - Deterministic Sobol global candidates + trust-region MIXTURE around top-k observed points
#  - Adaptive trust radius based on local spacing (more reproducible than fixed radius)
#  - Novelty filtering via NearestNeighbors (fast + deterministic)
#  - Fix "singleton normalize" issue in refinement (transparent selection: robust EI primary, UCB tie-break)
#  - Decision log printed for reproducibility and interpretability
#  - Always prints x_next with <= 6 decimals
# ============================================================

# ============================================================
# 0. Global settings
# ============================================================

RANDOM_STATE = 123
DEVICE = torch.device("cpu")  # "cuda" if available

# ---- Stage A (cheap) ----
STAGE_A_TUNE_TRIALS = 10
STAGE_A_KFOLDS = 3
STAGE_A_EPOCHS = 350
STAGE_A_ENSEMBLE = (5, 7, 9)

# ---- Stage B (expensive, only once) ----
STAGE_B_EPOCHS = 1400
STAGE_B_ENSEMBLE_MIN = 10
STAGE_B_ENSEMBLE_MAX = 16

# Candidate search scaling
FINAL_N_CANDIDATES = 120_000
FINAL_N_RESTARTS = 18
NOVELTY_MIN_L2 = 0.015
NOVELTY_MAX_TRIES = 8

# Robust acquisition / emergent-behaviour hedge
XI_GRID = [0.0, 0.005, 0.01, 0.02]   # average EI across these
ACQ_BLEND_EI = 0.65                  # EI weight
ACQ_BLEND_UCB = 0.35                 # UCB weight

# Trust region mixture (Week 10)
TOPK_ANCHORS_DEFAULT = 5             # how many top observed points to anchor around
TRUST_RADIUS_MIN = 0.04
TRUST_RADIUS_MAX = 0.14
TRUST_RADIUS_SPACING_MULT = 0.35     # trust_radius = mult * median(nn spacing among top-k)

# Numerical stability
EPS_SIGMA = 1e-9

# Early stopping (cost/robustness trade-off)
EARLY_STOP_PATIENCE = 40
EARLY_STOP_MIN_DELTA = 1e-5

# ============================================================
# 1. Data (as provided)
# ============================================================

X_raw = np.array([
 [0.60499445, 0.29221502, 0.90845275, 0.35550624, 0.20166872, 0.57533801, 0.31031095, 0.73428138],
 [0.17800696, 0.56622265, 0.99486184, 0.21032501, 0.32015266, 0.70790879, 0.63538449, 0.10713163],
 [0.00907698, 0.81162615, 0.52052036, 0.07568668, 0.26511183, 0.09165169, 0.59241515, 0.36732026],
 [0.50602816, 0.65373012, 0.36341078, 0.17798105, 0.0937283, 0.19742533, 0.7558269, 0.29247234],
 [0.35990926, 0.24907568, 0.49599717, 0.70921498, 0.11498719, 0.28920692, 0.55729515, 0.59388173],
 [0.77881834, 0.0034195, 0.33798313, 0.51952778, 0.82090699, 0.53724669, 0.5513471, 0.66003209],
 [0.90864932, 0.0622497, 0.23825955, 0.76660355, 0.13233596, 0.99024381, 0.68806782, 0.74249594],
 [0.58637144, 0.88073573, 0.74502075, 0.54603485, 0.00964888, 0.74899176, 0.23090707, 0.09791562],
 [0.76113733, 0.85467239, 0.38212433, 0.33735198, 0.68970832, 0.30985305, 0.63137968, 0.04195607],
 [0.9849332, 0.69950626, 0.9988855, 0.18014846, 0.58014315, 0.23108719, 0.49082694, 0.31368272],
 [0.11207131, 0.43773566, 0.59659878, 0.59277563, 0.22698177, 0.41010452, 0.92123758, 0.67475276],
 [0.79188751, 0.57619134, 0.69452836, 0.28342378, 0.13675546, 0.27916186, 0.84276726, 0.62532792],
 [0.1435503, 0.93741452, 0.23232482, 0.00904349, 0.41457893, 0.40932517, 0.55377852, 0.2058408],
 [0.76991655, 0.45875909, 0.55900044, 0.69460444, 0.50319902, 0.72834638, 0.78425353, 0.66313109],
 [0.05644741, 0.06595555, 0.02292868, 0.03878647, 0.40393544, 0.80105533, 0.48830701, 0.89308498],
 [0.86243745, 0.48273382, 0.2818694, 0.54410223, 0.88749026, 0.38265469, 0.60190199, 0.47646169],
 [0.3515119, 0.59006494, 0.9094363, 0.67840835, 0.21282566, 0.08846038, 0.410153, 0.19572429],
 [0.73590364, 0.03461189, 0.72803027, 0.14742652, 0.29574314, 0.44511731, 0.97517969, 0.37433978],
 [0.68029397, 0.25510465, 0.86218799, 0.13439582, 0.3263292, 0.28790687, 0.43501048, 0.36420013],
 [0.04432925, 0.01358149, 0.25819824, 0.57764416, 0.05127992, 0.15856307, 0.59103012, 0.07795293],
 [0.77834548, 0.75114565, 0.31414221, 0.90298577, 0.33538166, 0.38632267, 0.74897249, 0.9887551],
 [0.89888711, 0.5236417, 0.87678325, 0.21869645, 0.90026089, 0.28276624, 0.91107791, 0.47239822],
 [0.14512029, 0.11932754, 0.42088822, 0.38760861, 0.15542283, 0.87517163, 0.51055967, 0.72861058],
 [0.33895442, 0.56693202, 0.3767511, 0.09891573, 0.65945169, 0.24554809, 0.76248278, 0.73215347],
 [0.17615002, 0.29396143, 0.97567997, 0.79393631, 0.92340076, 0.03084229, 0.80325452, 0.59589758],
 [0.02894663, 0.02827906, 0.48137155, 0.6131746, 0.67266045, 0.02211341, 0.6014833, 0.52488505],
 [0.19263987, 0.63067728, 0.41679584, 0.49052929, 0.79608602, 0.65456706, 0.27624119, 0.29551759],
 [0.94318502, 0.21885062, 0.72118408, 0.42459707, 0.986902, 0.53518298, 0.71474318, 0.96009372],
 [0.5327214, 0.8336926, 0.071399, 0.11681148, 0.73069311, 0.93737559, 0.86650798, 0.127902],
 [0.44709584, 0.84395253, 0.72954612, 0.63915138, 0.40928714, 0.13264569, 0.03590888, 0.44683847],
 [0.38222497, 0.55713584, 0.85310163, 0.33379569, 0.26572127, 0.48087292, 0.23764706, 0.76863196],
 [0.53281953, 0.86230848, 0.53826712, 0.04944293, 0.71970119, 0.9067059, 0.10823094, 0.52534791],
 [0.39486519, 0.33180167, 0.7407543, 0.69786172, 0.73740444, 0.78377681, 0.25449546, 0.87114551],
 [0.98594539, 0.87305363, 0.07039262, 0.05358729, 0.73415296, 0.52025852, 0.81104004, 0.10336036],
 [0.96457339, 0.97397979, 0.66375335, 0.66221599, 0.67312167, 0.90523762, 0.45887462, 0.5609175],
 [0.47207071, 0.16820264, 0.08642757, 0.45265551, 0.48061922, 0.62243949, 0.92897446, 0.11253627],
 [0.85600695, 0.6388937, 0.32619202, 0.66850311, 0.24029837, 0.21029889, 0.16754636, 0.96358986],
 [0.81003174, 0.63504604, 0.26954758, 0.86960534, 0.66192159, 0.25225873, 0.76567003, 0.89054867],
 [0.79625252, 0.00703653, 0.35569738, 0.48756605, 0.74051962, 0.7066501, 0.99291449, 0.38173437],
 [0.48124533, 0.10246072, 0.21948594, 0.67732237, 0.24750919, 0.24434086, 0.16382453, 0.71596164],
 [1.085945, 1.073979, 1.098885, 1.002985, 1.086901, 1.090243, 1.092914, 1.092915],
 [1.00000e-06, 2.05513e-01, 1.00000e-06, 8.42570e-02, 8.08305e-01, 3.55768e-01, 1.00000e-06, 4.52510e-01],
 [0.025554, 0.341676, 0.022591, 0.351417, 0.540332, 0.56768 , 0.214489, 0.595807],
 [0.157856, 0.176691, 0.024491, 0.477665, 0.906327, 0.715038, 0.022394, 0.250892],
 [0.026581, 0.517116, 0.045489, 0.164555, 0.986197, 0.391204, 0.022672, 0.723886],
 [0.      , 0.171985, 0.      , 0.282222, 1.      , 1.      , 0.      , 1.      ],
 [0.      , 0.68071 , 0.      , 0.073419, 1.      , 0.509956, 0.      , 0.      ],
 [0.000000, 0.000000, 0.000000, 0.444056, 0.81141 , 0.577367,0.000000, 1.000000],
 [0.000000, 0.229161, 0.000000, 0.909293, 0.921604, 0.631610, 0.000000, 0.000000]
])

y_raw = np.array([
    7.3987211, 7.00522736, 8.45948162, 8.28400781, 8.60611679,
    8.54174792, 7.32743458, 7.29987205, 7.95787474, 5.59219339,
    7.85454099, 6.79198578, 8.97655402, 7.3790829,  9.598482,
    8.15998319, 7.13162397, 6.76796253, 7.43374407, 9.01307515,
    7.31089382, 5.84106731, 9.14163949, 8.81755844, 6.45194313,
    8.83074505, 9.34427428, 6.88784639, 8.04221254, 7.69236805,
    7.92375877, 8.42175924, 8.2780624,  7.11345716, 6.40258841,
    8.47293632, 7.97768459, 7.46087219, 7.43659353, 9.18300525,
    1.6498596481450019, 9.8188854584285, 9.8382810715811,
    9.7246626786321, 9.7392105237459, 9.545334002491, 9.505683124403, 9.718280322125,
    9.205792717722
])

assert X_raw.shape[0] == y_raw.shape[0], "X and y sizes must match"

# ============================================================
# 2. PyTorch Deep Ensemble Surrogate
# ============================================================

class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim: int, hidden_dims=(128, 128, 64), dropout=0.0):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(p=dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def build_ensemble_torch(n_members: int, input_dim: int, hidden_dims, dropout) -> List[MLPRegressorTorch]:
    ensemble = []
    for m in range(n_members):
        torch.manual_seed(RANDOM_STATE + 10_000 + m)
        model = MLPRegressorTorch(input_dim, hidden_dims=hidden_dims, dropout=dropout).to(DEVICE)
        ensemble.append(model)
    return ensemble

def fit_ensemble_torch(
    ensemble: List[MLPRegressorTorch],
    X_scaled: np.ndarray,
    y_scaled: np.ndarray,
    n_epochs: int,
    batch_size: int,
    lr: float,
    weight_decay: float,
) -> None:
    rng = np.random.RandomState(RANDOM_STATE + 1)
    X = torch.from_numpy(X_scaled.astype(np.float32)).to(DEVICE)
    y = torch.from_numpy(y_scaled.astype(np.float32)).view(-1, 1).to(DEVICE)
    n_samples = X.shape[0]

    for model in ensemble:
        idx = rng.randint(0, n_samples, size=n_samples)
        X_boot = X[idx]
        y_boot = y[idx]

        dataset = TensorDataset(X_boot, y_boot)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.SmoothL1Loss()

        model.train()
        best_loss = np.inf
        bad = 0

        for _ in range(n_epochs):
            epoch_loss = 0.0
            n_seen = 0
            for xb, yb in loader:
                optimizer.zero_grad()
                preds = model(xb)
                loss = criterion(preds, yb)
                loss.backward()
                optimizer.step()
                epoch_loss += float(loss.detach().cpu().item()) * xb.shape[0]
                n_seen += xb.shape[0]
            epoch_loss /= max(1, n_seen)

            # Early stopping
            if epoch_loss < best_loss - EARLY_STOP_MIN_DELTA:
                best_loss = epoch_loss
                bad = 0
            else:
                bad += 1
                if bad >= EARLY_STOP_PATIENCE:
                    break

def ensemble_predict_torch(ensemble: List[MLPRegressorTorch], X_scaled: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X_t = torch.from_numpy(X_scaled.astype(np.float32)).to(DEVICE)
    preds_list = []
    for model in ensemble:
        model.eval()
        with torch.no_grad():
            preds = model(X_t).cpu().numpy().ravel()
        preds_list.append(preds)
    preds = np.stack(preds_list, axis=0)
    mu = preds.mean(axis=0)
    sigma = preds.std(axis=0, ddof=1) + EPS_SIGMA
    return mu, sigma

# ============================================================
# 3. Acquisition functions
# ============================================================

def expected_improvement(mu: np.ndarray, sigma: np.ndarray, best_y: float, xi: float) -> np.ndarray:
    sigma_safe = np.maximum(sigma, 1e-12)
    z = (mu - best_y - xi) / sigma_safe
    ei = (mu - best_y - xi) * norm.cdf(z) + sigma_safe * norm.pdf(z)
    ei[sigma_safe < 1e-12] = 0.0
    return ei

def ucb(mu: np.ndarray, sigma: np.ndarray, beta: float) -> np.ndarray:
    return mu + beta * sigma

def normalize01(a: np.ndarray) -> np.ndarray:
    mn = float(np.min(a))
    mx = float(np.max(a))
    if mx - mn < 1e-12:
        return np.zeros_like(a)
    return (a - mn) / (mx - mn)

# ============================================================
# 4. EI and gradient for a single point (PyTorch autograd)
# ============================================================

def ei_and_grad_single(
    x_raw_1d: np.ndarray,
    ensemble: List[MLPRegressorTorch],
    x_scaler: MinMaxScaler,
    y_scaler: StandardScaler,
    best_y: float,
    xi: float,
) -> Tuple[float, np.ndarray]:
    x_raw = np.clip(x_raw_1d.reshape(1, -1), 0.0, 1.0)
    x_scaled = x_scaler.transform(x_raw)
    x_t = torch.tensor(x_scaled.astype(np.float32), requires_grad=True, device=DEVICE)

    preds = []
    for model in ensemble:
        model.eval()
        preds.append(model(x_t))

    Y = torch.stack(preds, dim=0).squeeze(-1).squeeze(-1)
    mu_scaled = Y.mean()
    sigma_scaled = Y.std(unbiased=True) + EPS_SIGMA

    scale_y = float(y_scaler.scale_[0])
    mean_y = float(y_scaler.mean_[0])
    mu_y = mu_scaled * scale_y + mean_y
    sigma_y = torch.clamp(sigma_scaled * scale_y, min=1e-8)

    normal = torch.distributions.Normal(0.0, 1.0)
    z = (mu_y - best_y - xi) / sigma_y
    cdf = normal.cdf(z)
    pdf = torch.exp(normal.log_prob(z))
    ei = (mu_y - best_y - xi) * cdf + sigma_y * pdf

    ei.backward()
    grad_x_scaled = x_t.grad.detach().cpu().numpy().reshape(-1)

    # x_scaled = x_raw * scale_ + min_ (MinMaxScaler); dx_scaled/dx_raw = scale_
    grad_x_raw = grad_x_scaled * x_scaler.scale_.astype(np.float32)
    return float(ei.detach().cpu().item()), grad_x_raw

# ============================================================
# 5. Lightweight tuning (Stage A only)
# ============================================================

def sample_config(rng: np.random.RandomState) -> Dict[str, Any]:
    hidden_choices = [
        (64, 64),
        (128, 64),
        (128, 128, 64),
        (256, 128, 64),
    ]
    return {
        "hidden_dims": hidden_choices[rng.randint(len(hidden_choices))],
        "dropout": float(rng.choice([0.0, 0.05, 0.1, 0.2])),
        "lr": float(10 ** rng.uniform(-3.4, -2.2)),
        "weight_decay": float(10 ** rng.uniform(-6.2, -3.8)),
        "batch_size": int(rng.choice([8, 16, 32])),
        "n_ensemble": int(rng.choice(list(STAGE_A_ENSEMBLE))),
    }

def cv_mse_for_config(
    cfg: Dict[str, Any],
    X_scaled: np.ndarray,
    y_scaled: np.ndarray,
    input_dim: int,
    n_epochs: int,
    n_folds: int,
) -> float:
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    fold_mse = []

    for tr_idx, va_idx in kf.split(X_scaled):
        X_tr, X_va = X_scaled[tr_idx], X_scaled[va_idx]
        y_tr, y_va = y_scaled[tr_idx], y_scaled[va_idx]

        ensemble = build_ensemble_torch(
            n_members=cfg["n_ensemble"],
            input_dim=input_dim,
            hidden_dims=cfg["hidden_dims"],
            dropout=cfg["dropout"],
        )

        fit_ensemble_torch(
            ensemble=ensemble,
            X_scaled=X_tr,
            y_scaled=y_tr,
            n_epochs=n_epochs,
            batch_size=cfg["batch_size"],
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
        )

        mu_va, _ = ensemble_predict_torch(ensemble, X_va)
        mse = mean_squared_error(y_va, mu_va)
        fold_mse.append(mse)

    return float(np.mean(fold_mse))

def tune_stage_a(X_raw_in: np.ndarray, y_raw_in: np.ndarray) -> Dict[str, Any]:
    rng = np.random.RandomState(RANDOM_STATE + 99)

    x_scaler = MinMaxScaler(feature_range=(0.0, 1.0))
    X_scaled = x_scaler.fit_transform(X_raw_in)

    y_scaler = StandardScaler()
    y_scaled = y_scaler.fit_transform(y_raw_in.reshape(-1, 1)).ravel()

    input_dim = X_raw_in.shape[1]

    best_cfg = None
    best_mse = np.inf
    for _ in range(STAGE_A_TUNE_TRIALS):
        cfg = sample_config(rng)
        mse = cv_mse_for_config(cfg, X_scaled, y_scaled, input_dim, n_epochs=STAGE_A_EPOCHS, n_folds=STAGE_A_KFOLDS)
        if mse < best_mse:
            best_mse = mse
            best_cfg = cfg

    assert best_cfg is not None
    best_cfg["n_ensemble_final"] = int(np.clip(best_cfg["n_ensemble"] + 6, STAGE_B_ENSEMBLE_MIN, STAGE_B_ENSEMBLE_MAX))
    best_cfg["n_epochs_final"] = int(STAGE_B_EPOCHS)
    best_cfg["cv_mse_stage_a"] = float(best_mse)
    return best_cfg

# ============================================================
# 6. BO result dataclass
# ============================================================

@dataclass
class BOResult:
    x_next: np.ndarray
    mu_next: float
    sigma_next: float
    acq_score: float
    x_best_observed: np.ndarray
    y_best_observed: float
    tuned_config: Dict[str, Any]
    reasoning: str

# ============================================================
# 7. Propose next query point (robust acquisition + trust mixture)
# ============================================================

def _min_l2_distance(x: np.ndarray, X: np.ndarray) -> float:
    d = np.linalg.norm(X - x.reshape(1, -1), axis=1)
    return float(np.min(d))

def propose_next_point(
    ensemble: List[MLPRegressorTorch],
    x_scaler: MinMaxScaler,
    y_scaler: StandardScaler,
    X_raw_in: np.ndarray,
    y_raw_in: np.ndarray,
    n_candidates: int,
    n_restarts: int,
    random_seed: int,
) -> BOResult:

    rng = np.random.RandomState(random_seed)
    best_idx = int(np.argmax(y_raw_in))
    x_best_obs = X_raw_in[best_idx]
    y_best_obs = float(y_raw_in[best_idx])
    dim = X_raw_in.shape[1]

    # ----- Candidate pool: Sobol global + trust-region mixture around top-k observed -----
    sampler = qmc.Sobol(d=dim, scramble=True, seed=random_seed)
    X_cand_raw = sampler.random(n_candidates).astype(np.float64)

    TOPK_ANCHORS = min(TOPK_ANCHORS_DEFAULT, len(y_raw_in))
    anchor_idx = np.argsort(-y_raw_in)[:TOPK_ANCHORS]
    anchors = X_raw_in[anchor_idx]

    # Adaptive trust radius based on local spacing among top-k anchors (more interpretable than a fixed constant)
    nn_top = NearestNeighbors(n_neighbors=min(2, len(X_raw_in))).fit(X_raw_in)
    # If we can't compute 2-neighbor distances (degenerate case), fall back
    if len(X_raw_in) >= 2:
        dists_top, _ = nn_top.kneighbors(anchors, n_neighbors=2, return_distance=True)
        typical_spacing = float(np.median(dists_top[:, 1]))
    else:
        typical_spacing = 0.10
    trust_radius = float(np.clip(TRUST_RADIUS_SPACING_MULT * typical_spacing, TRUST_RADIUS_MIN, TRUST_RADIUS_MAX))

    n_tr = max(12_000, n_candidates // 6)
    mix_ids = rng.randint(0, TOPK_ANCHORS, size=n_tr)
    tr = anchors[mix_ids] + trust_radius * rng.randn(n_tr, dim)
    tr = np.clip(tr, 0.0, 1.0)
    X_cand_raw[:n_tr] = tr

    # ----- Novelty filter via NN (fast + deterministic) -----
    nn = NearestNeighbors(n_neighbors=1, algorithm="kd_tree").fit(X_raw_in)
    dist, _ = nn.kneighbors(X_cand_raw)
    dist = dist.ravel()
    mask = dist >= NOVELTY_MIN_L2
    if np.any(mask):
        X_cand_raw = X_cand_raw[mask]
        dist = dist[mask]

    # Predict
    X_cand_scaled = x_scaler.transform(X_cand_raw)
    mu_scaled, sigma_scaled = ensemble_predict_torch(ensemble, X_cand_scaled)
    mu = y_scaler.inverse_transform(mu_scaled.reshape(-1, 1)).ravel()
    sigma = sigma_scaled * y_scaler.scale_[0]

    # Robust EI: average across xi grid
    eis = np.stack([expected_improvement(mu, sigma, y_best_obs, xi=xi) for xi in XI_GRID], axis=0)
    ei_avg = np.mean(eis, axis=0)

    # UCB: beta grows mildly with dim and log(n)
    beta = float(1.6 + 0.15 * dim + 0.30 * np.log(1 + len(y_raw_in)))
    ucb_score = ucb(mu, sigma, beta=beta)

    # Blend (normalized over the full candidate set)
    score = ACQ_BLEND_EI * normalize01(ei_avg) + ACQ_BLEND_UCB * normalize01(ucb_score)

    # Seeds for refinement
    top_indices = np.argsort(-score)[:n_restarts]
    x_seed = X_cand_raw[int(top_indices[0])].copy()

    # We'll refine using EI with a middle xi (stable gradients)
    xi_refine = 0.01
    bounds = [(0.0, 1.0)] * dim

    cache_x: Optional[np.ndarray] = None
    cache_val: Optional[float] = None
    cache_grad: Optional[np.ndarray] = None

    def objective_and_grad(x: np.ndarray) -> Tuple[float, np.ndarray]:
        nonlocal cache_x, cache_val, cache_grad
        if cache_x is not None and np.allclose(x, cache_x, atol=1e-12, rtol=0.0):
            return cache_val, cache_grad
        ei_val, grad = ei_and_grad_single(
            x_raw_1d=x,
            ensemble=ensemble,
            x_scaler=x_scaler,
            y_scaler=y_scaler,
            best_y=y_best_obs,
            xi=xi_refine,
        )
        cache_x = x.copy()
        cache_val = -ei_val
        cache_grad = -grad
        return cache_val, cache_grad

    best_x = x_seed.copy()

    # Week 10 transparent refinement selection:
    # 1) maximize robust EI_avg
    # 2) tie-break on UCB
    best_ei = -np.inf
    best_ucb = -np.inf

    for idx in top_indices:
        x0 = X_cand_raw[int(idx)]
        cache_x = None
        res = minimize(
            fun=lambda x: objective_and_grad(x)[0],
            x0=x0,
            method="L-BFGS-B",
            jac=lambda x: objective_and_grad(x)[1],
            bounds=bounds,
            options={"maxiter": 160},
        )
        if not res.success:
            continue

        x_try = np.clip(res.x, 0.0, 1.0)
        x_try_scaled = x_scaler.transform(x_try.reshape(1, -1))
        mu_s, sig_s = ensemble_predict_torch(ensemble, x_try_scaled)
        mu_t = float(y_scaler.inverse_transform(mu_s.reshape(-1, 1))[0, 0])
        sig_t = float(sig_s[0] * y_scaler.scale_[0])

        ei_try = float(np.mean([
            expected_improvement(np.array([mu_t]), np.array([sig_t]), y_best_obs, xi)[0]
            for xi in XI_GRID
        ]))
        ucb_try = float(mu_t + beta * sig_t)

        if (ei_try > best_ei + 1e-12) or (abs(ei_try - best_ei) <= 1e-12 and ucb_try > best_ucb):
            best_ei = ei_try
            best_ucb = ucb_try
            best_x = x_try.copy()

    # Ensure novelty with jitter if needed
    for _ in range(NOVELTY_MAX_TRIES):
        if _min_l2_distance(best_x, X_raw_in) >= NOVELTY_MIN_L2:
            break
        best_x = np.clip(best_x + 0.02 * rng.randn(dim), 0.0, 1.0)

    # Final prediction at best_x
    x_final_scaled = x_scaler.transform(best_x.reshape(1, -1))
    mu_scaled_f, sigma_scaled_f = ensemble_predict_torch(ensemble, x_final_scaled)
    mu_f = float(y_scaler.inverse_transform(mu_scaled_f.reshape(-1, 1))[0, 0])
    sigma_f = float(sigma_scaled_f[0] * y_scaler.scale_[0])

    # Final reported acquisition score = robust EI avg
    ei_f = float(np.mean([expected_improvement(np.array([mu_f]), np.array([sigma_f]), y_best_obs, xi)[0] for xi in XI_GRID]))

    decision_log = {
        "y_best": round(y_best_obs, 6),
        "x_best": np.round(x_best_obs, 6).tolist(),
        "trust_radius_used": float(trust_radius),
        "topk_anchor_indices": anchor_idx.tolist(),
        "beta": float(beta),
        "xi_grid": list(XI_GRID),
        "acq_blend": {"EI": float(ACQ_BLEND_EI), "UCB": float(ACQ_BLEND_UCB)},
        "novelty_min_l2": float(NOVELTY_MIN_L2),
        "x_next_nn_dist": float(_min_l2_distance(best_x, X_raw_in)),
        "mu_next": float(mu_f),
        "sigma_next": float(sigma_f),
        "robust_ei_next": float(ei_f),
        "n_candidates_after_novelty": int(X_cand_raw.shape[0]),
    }

    reasoning = "\n".join([
        f"Observed best: y_best = {y_best_obs:.6f} at x_best = {np.round(x_best_obs, 6)}.",
        "Week 10 transparency: deterministic Sobol candidates + logged decision variables.",
        f"Candidate pool: Sobol global + trust-mixture around top-{TOPK_ANCHORS} observed points; adaptive radius={trust_radius:.6f}.",
        f"Robust acquisition: EI averaged across xi={XI_GRID} blended with UCB (beta≈{beta:.3f}).",
        f"Novelty: NN distance ≥ {NOVELTY_MIN_L2} (chosen NN dist={decision_log['x_next_nn_dist']:.6f}).",
        f"Refinement: L-BFGS-B from top {n_restarts} seeds using EI gradient at xi={xi_refine}.",
        f"Final: x_next≈{np.round(best_x, 6)}, μ≈{mu_f:.6f}, σ≈{sigma_f:.6f}, robust_EI≈{ei_f:.6f}.",
        "Selection rule: maximize robust_EI; tie-break on UCB (transparent + avoids singleton normalization bug).",
        "Decision log (reproducible): " + str(decision_log),
    ])

    return BOResult(
        x_next=best_x,
        mu_next=mu_f,
        sigma_next=sigma_f,
        acq_score=ei_f,
        x_best_observed=x_best_obs,
        y_best_observed=y_best_obs,
        tuned_config={},
        reasoning=reasoning,
    )

# ============================================================
# 8. Main script (Week 10)
# ============================================================

def fmt_x6(x: np.ndarray) -> str:
    x6 = np.round(np.clip(x, 0.0, 1.0), 6)
    return np.array2string(x6, precision=6, separator=", ", suppress_small=False)

def main():
    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)

    print("======================================================")
    print("WEEK 10 FUNCTION 8 — TRANSPARENT + ROBUST BO (8D)")
    print("======================================================\n")

    # Enforce domain [0,1]^8 for training stability
    X_train = np.clip(X_raw, 0.0, 1.0)
    n_oob = int(np.sum((X_raw < 0.0) | (X_raw > 1.0)))
    if n_oob > 0:
        print(f"[Note] Detected {n_oob} out-of-bound X entries; clipped to [0,1].\n")

    print("Stage A: lightweight tuning (controls compute; avoids diminishing returns)...")
    best_cfg = tune_stage_a(X_train, y_raw)
    print("Best Stage A config:")
    for k, v in best_cfg.items():
        print(f"  {k}: {v}")
    print()

    # Scale inputs and outputs
    x_scaler = MinMaxScaler(feature_range=(0.0, 1.0))
    X_scaled = x_scaler.fit_transform(X_train)

    y_scaler = StandardScaler()
    y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

    # Stage B: final surrogate with scaled-up ensemble/epochs
    print("Stage B: training FINAL ensemble surrogate (scaled compute where it matters)...")
    input_dim = X_train.shape[1]
    ensemble = build_ensemble_torch(
        n_members=best_cfg["n_ensemble_final"],
        input_dim=input_dim,
        hidden_dims=best_cfg["hidden_dims"],
        dropout=best_cfg["dropout"],
    )
    fit_ensemble_torch(
        ensemble=ensemble,
        X_scaled=X_scaled,
        y_scaled=y_scaled,
        n_epochs=best_cfg["n_epochs_final"],
        batch_size=best_cfg["batch_size"],
        lr=best_cfg["lr"],
        weight_decay=best_cfg["weight_decay"],
    )
    print("Training complete.\n")

    # Propose next point
    result = propose_next_point(
        ensemble=ensemble,
        x_scaler=x_scaler,
        y_scaler=y_scaler,
        X_raw_in=X_train,
        y_raw_in=y_raw,
        n_candidates=FINAL_N_CANDIDATES,
        n_restarts=FINAL_N_RESTARTS,
        random_seed=RANDOM_STATE + 5,
    )
    result.tuned_config = best_cfg

    print("=== CURRENT BEST OBSERVED ===")
    print(f"x_best = {fmt_x6(result.x_best_observed)}")
    print(f"y_best = {result.y_best_observed:.6f}\n")

    print("=== RECOMMENDED NEXT POINT (<= 6 decimals) ===")
    print(f"x_next         = {fmt_x6(result.x_next)}")
    print(f"μ(x_next)      = {result.mu_next:.6f}")
    print(f"σ(x_next)      = {result.sigma_next:.6f}")
    print(f"robust_EI_avg  = {result.acq_score:.6f}\n")

    print("=== REASONING ===")
    print(result.reasoning)
    print("\nDone.")

if __name__ == "__main__":
    main()


WEEK 10 FUNCTION 8 — TRANSPARENT + ROBUST BO (8D)

[Note] Detected 8 out-of-bound X entries; clipped to [0,1].

Stage A: lightweight tuning (controls compute; avoids diminishing returns)...
Best Stage A config:
  hidden_dims: (256, 128, 64)
  dropout: 0.0
  lr: 0.0006772190434264009
  weight_decay: 1.0582518184027805e-06
  batch_size: 16
  n_ensemble: 7
  n_ensemble_final: 13
  n_epochs_final: 1400
  cv_mse_stage_a: 0.20702545005247178

Stage B: training FINAL ensemble surrogate (scaled compute where it matters)...
Training complete.



C:\Anaconda3\Lib\site-packages\scipy\stats\_qmc.py:993: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  sample = self._random(n, workers=workers)


=== CURRENT BEST OBSERVED ===
x_best = [0.025554, 0.341676, 0.022591, 0.351417, 0.540332, 0.56768 , 0.214489,
 0.595807]
y_best = 9.838281

=== RECOMMENDED NEXT POINT (<= 6 decimals) ===
x_next         = [0.      , 0.406814, 0.      , 0.537937, 0.951392, 1.      , 0.      ,
 1.      ]
μ(x_next)      = 9.711928
σ(x_next)      = 0.199744
robust_EI_avg  = 0.029742

=== REASONING ===
Observed best: y_best = 9.838281 at x_best = [0.025554 0.341676 0.022591 0.351417 0.540332 0.56768  0.214489 0.595807].
Week 10 transparency: deterministic Sobol candidates + logged decision variables.
Candidate pool: Sobol global + trust-mixture around top-5 observed points; adaptive radius=0.140000.
Robust acquisition: EI averaged across xi=[0.0, 0.005, 0.01, 0.02] blended with UCB (beta≈3.974).
Novelty: NN distance ≥ 0.015 (chosen NN dist=0.350568).
Refinement: L-BFGS-B from top 18 seeds using EI gradient at xi=0.01.
Final: x_next≈[0.       0.406814 0.       0.537937 0.951392 1.       0.       1.      ], μ≈